In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
import re

In [2]:
# Load Dataset
df = pd.read_csv("data.csv")

# Clean column names (strip spaces)
df.columns = df.columns.str.strip()
df.head()

,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR),has_creditcard,credit_cardtype,has_active_loan,Loan_type,savings_plan_type,investment_type
0,T1,C5841053,10/1/1994,F,JAMSHEDPUR,17819.05,2/8/2016,143207,25.0,0,NaN,1,Car,Premium,Stocks
1,T2,C2142763,4/4/1957,M,JHAJJAR,2270.69,2/8/2016,141858,27999.0,0,NaN,0,NaN,Basic,NaN
2,T3,C4417068,26/11/96,F,MUMBAI,17874.44,2/8/2016,142712,459.0,0,NaN,0,NaN,Basic,Stocks
3,T4,C5342380,14/9/73,F,MUMBAI,866503.21,2/8/2016,142714,2060.0,1,Platinum,0,NaN,Premium,NaN
4,T5,C9031234,24/3/88,F,NAVI MUMBAI,6714.43,2/8/2016,181156,1762.5,1,Silver,0,NaN,Premium,Bonds


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            1000 non-null   object 
 1   CustomerID               1000 non-null   object 
 2   CustomerDOB              999 non-null    object 
 3   CustGender               999 non-null    object 
 4   CustLocation             1000 non-null   object 
 5   CustAccountBalance       999 non-null    float64
 6   TransactionDate          1000 non-null   object 
 7   TransactionTime          1000 non-null   int64  
 8   TransactionAmount (INR)  1000 non-null   float64
 9   has_creditcard           1000 non-null   int64  
 10  credit_cardtype          350 non-null    object 
 11  has_active_loan          1000 non-null   int64  
 12  Loan_type                475 non-null    object 
 13  savings_plan_type        1000 non-null   object 
 14  investment_type          

In [4]:
# Identify Features
numeric_features = df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_features = df.select_dtypes(include=['object']).columns.tolist()

# Remove customer ID and target columns from features
for col in ['CustomerID', 'Loan_type', 'credit_cardtype', 'investment_type', 'savings_plan_type']:
    if col in numeric_features:
        numeric_features.remove(col)
    if col in cat_features:
        cat_features.remove(col)

In [5]:
# Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, cat_features)
    ]
)


In [6]:
#  Customer Segmentation (Clustering)
X_features = preprocessor.fit_transform(df[numeric_features + cat_features])
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X_features)

In [7]:
# Train Classifiers for Each Product
def train_classifier(target_col):
    y = df[target_col].astype(str).fillna('None')
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    clf.fit(df[numeric_features + cat_features], y_encoded)
    return clf, le

loan_clf, loan_le = train_classifier('Loan_type')
cc_clf, cc_le = train_classifier('credit_cardtype')
inv_clf, inv_le = train_classifier('investment_type')
savings_clf, savings_le = train_classifier('savings_plan_type')

In [8]:
# Dynamic Recommendation Function
def recommend(customer_ids, product_type='loan', top_n=3):
    if isinstance(customer_ids, str):
        customer_ids = [customer_ids]

    product_map = {
        'loan': (loan_clf, loan_le, 'Loan_type'),
        'credit_card ': (cc_clf, cc_le, 'credit_cardtype'),
        'investment': (inv_clf, inv_le, 'investment_type'),
        'savings': (savings_clf, savings_le, 'savings_plan_type')
    }
    
    if product_type not in product_map:
        return f"Product type must be one of {list(product_map.keys())}"

    clf, le, col_name = product_map[product_type]
    output = {}

    for cust_id in customer_ids:
        customer = df[df['CustomerID'] == cust_id]
        if customer.empty:
            output[cust_id] = f"Customer {cust_id} not found"
            continue
        
        X_cust = customer[numeric_features + cat_features]
        probs = clf.predict_proba(X_cust)[0]
        types = le.inverse_transform(np.arange(len(probs)))
        prob_df = pd.DataFrame({'type': types, 'probability': probs}).sort_values(by='probability', ascending=False).head(top_n)

        cluster_id = int(customer['cluster'].values[0])
        cluster_customers = df[df['cluster'] == cluster_id]
        # popular = cluster_customers[col_name].value_counts().head(top_n).to_dict()
        popular = list(cluster_customers[col_name].value_counts().head(top_n).index)
        suggestions_text = ", ".join(popular)
        output = f"Customer ID : {cust_id}\nFor your {product_type} we suggest : {suggestions_text}"
    return output


In [9]:
# Chat-Style Input Function
def chat_recommend(user_input):
    product_keywords = {
        'loan': ['loan','Personal','Car','Home','Education'],
        'credit_card': ['credit','credit card','Silver','Platinum','Gold'],
        'investment': ['investment','Mutual Fund','Bonds','Stocks'],
        'savings': ['savings','saving','Basic','Premium','Fixed Deposit']
    }
    
    product_type = None
    for key, keywords in product_keywords.items():
        for kw in keywords:
            if re.search(kw, user_input, re.IGNORECASE):
                product_type = key
                break
        if product_type:
            break
    if not product_type:
        return "Sorry, I couldn't detect the product type. Please mention loan, credit card, investment, or savings."

    customer_ids = re.findall(r'[Cc]\d+', user_input)
    if not customer_ids:
        return "Sorry, I couldn't detect any CustomerID in your input. Kindly provide the Product type with CustomerID"

    return recommend(customer_ids, product_type=product_type)


In [10]:
# Example Chat Usage
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Chat ended.")
        break
    response = chat_recommend(user_input)
    print("Bot:", response)

You:  suggest loan type for C2142763


Bot: Customer ID : C2142763
For your loan we suggest : Personal, Car, Education


You:  exit


Chat ended.


In [29]:
# Example Chat Usage
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Chat ended.")
        break
    response = chat_recommend(user_input)
    print("Bot:", response)

You:  suggest loan type for C2142763


Bot: Customer ID : C2142763
For your loan we suggest : Personal, Car, Education


You:  suggest a loan type for C4636775


Bot: Customer ID : C4636775
For your loan we suggest : Education, Personal, Car


You:  suggest investment type for C1728525


...

Bot: Customer ID : C1728525
For your loan we suggest : Education, Personal, Car


You:  suggest savings type for C1728525


Bot: Customer ID : C1728525
For your savings we suggest : Fixed Deposit, Basic, Premium


You:  exit


Chat ended.


In [13]:
# Example Chat Usage
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Chat ended.")
        break
    response = chat_recommend(user_input)
    print("Bot:", response)

You:  suggest loan type


Bot: Sorry, I couldn't detect any CustomerID in your input. Kindly provide the Product type with CustomerID


You:  C1728525


Bot: Sorry, I couldn't detect the product type. Please mention loan, credit card, investment, or savings.


You:  suggest loan type for C1728525


Bot: Customer ID : C1728525
For your loan we suggest : Education, Personal, Car


You:  exit


Chat ended.
